# CS2309 — SwiftEdit: notebook test

Notebook chạy thử **text-guided image editing** (SwiftEdit) trên **Mac MPS** hoặc **Google Colab (CUDA)**.

### Mac (local)
1. `pyenv local 3.12.10` → `.venv` → `pip install -r requirements-mac.txt`
2. `bash scripts/download_swiftedit_weights.sh` + `bash scripts/download_hf_models.sh`
3. Kernel **`.venv` (Python 3.12)**

### Google Colab (T4) — **không cần Google Drive**
1. Runtime → **Change runtime type → T4 GPU**
2. Chạy lần lượt các cell — notebook **tự nhận Colab**, **clone repo đề tài** `CS2309.CH201` (đã có `SwiftEdit/` + patch), tải weights/HF vào **`/content`** (mất khi reset runtime)
3. Chỉnh `REPO_URL` ở cell 1 nếu fork repo khác

**Lưu ý:** Không clone repo Qualcomm riêng — dùng `SwiftEdit/` trong repo đề tài.

In [1]:
import os
import subprocess
import sys
import time
from pathlib import Path

# Tự nhận Colab (hoặc gán tay IN_COLAB = True/False)
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- Colab: chỉ clone repo đề tài → lưu trên /content (không mount Drive) ---
REPO_URL = "https://github.com/NguyenKz/CS2309.CH201.git"  # đổi nếu dùng fork
COLAB_REPO_DIR = Path("/content/CS2309.CH201")

if IN_COLAB:
    if not (COLAB_REPO_DIR / "SwiftEdit" / "infer.py").exists():
        print(f"Cloning {REPO_URL} → {COLAB_REPO_DIR} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
    PROJECT_ROOT = COLAB_REPO_DIR
    # Cache Hugging Face trên ổ Colab (không Drive)
    os.environ.setdefault("HF_HOME", "/content/huggingface")
    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "900")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif not ((PROJECT_ROOT / "SwiftEdit" / "infer.py").exists()):
        for p in [Path.cwd(), *Path.cwd().parents]:
            if (p / "SwiftEdit" / "infer.py").exists():
                PROJECT_ROOT = p
                break

SWIFTEDIT_DIR = PROJECT_ROOT / "SwiftEdit"
WEIGHTS_DIR = SWIFTEDIT_DIR / "swiftedit_weights"
OUTPUT_DIR = PROJECT_ROOT / "results" / "notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(SWIFTEDIT_DIR)
if str(SWIFTEDIT_DIR) not in sys.path:
    sys.path.insert(0, str(SWIFTEDIT_DIR))

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SWIFTEDIT_DIR:", SWIFTEDIT_DIR)
print("HF_HOME:", os.environ.get("HF_HOME", "(default ~/.cache)"))
print("Weights OK:", (WEIGHTS_DIR / "inverse_ckpt-120k").is_dir())

PROJECT_ROOT: /Users/nguyenkz/Documents/code/CS2309.CH201
SWIFTEDIT_DIR: /Users/nguyenkz/Documents/code/CS2309.CH201/SwiftEdit
Weights OK: True


In [2]:
%pip install -q matplotlib ipywidgets

if IN_COLAB:
    # CUDA — requirements gốc trong SwiftEdit/ (repo đề tài)
    %pip install -q -r requirements.txt numpy==1.26.4

    os.chdir(PROJECT_ROOT)
    if not (WEIGHTS_DIR / "inverse_ckpt-120k").is_dir():
        print("Tải swiftedit_weights (~9.6 GB) → /content/CS2309.CH201/SwiftEdit/ ...")
        subprocess.run(["bash", "scripts/download_swiftedit_weights.sh"], check=True)
    else:
        print("swiftedit_weights: đã có, bỏ qua tải.")

    print("Tải model Hugging Face →", os.environ.get("HF_HOME", "/content/huggingface"))
    subprocess.run(["bash", "scripts/download_hf_models.sh"], check=True)
    os.chdir(SWIFTEDIT_DIR)
else:
    # Mac: cần tải weights trước (README § Cài đặt)
    pass

import torch

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("mps:", getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())

if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError("Colab chưa có GPU — Runtime → Change runtime type → T4 GPU")

if not (WEIGHTS_DIR / "inverse_ckpt-120k").is_dir():
    raise FileNotFoundError(
        f"Thiếu weights tại {WEIGHTS_DIR}\n"
        "Mac: bash scripts/download_swiftedit_weights.sh\n"
        "Colab: chạy lại cell này để tự tải."
    )

Note: you may need to restart the kernel to use updated packages.
torch: 2.12.0
cuda: False
mps: True


In [3]:
from infer import SWIFTEDIT_WEIGHTS_ROOT, edit_image, get_device
from models import AuxiliaryModel, IPSBV2Model, InverseModel

device = get_device()
print("Using device:", device)

inverse_ckpt = os.path.join(SWIFTEDIT_WEIGHTS_ROOT, "inverse_ckpt-120k")
path_unet_sb = os.path.join(SWIFTEDIT_WEIGHTS_ROOT, "sbv2_0.5")
ip_ckpt = os.path.join(SWIFTEDIT_WEIGHTS_ROOT, "ip_adapter_ckpt-90k/ip_adapter.bin")

print("Loading models (lần đầu có thể vài phút + tải HF)...")
t0 = time.time()
inverse_model = InverseModel(inverse_ckpt, device=device)
aux_model = AuxiliaryModel(device=device)
ip_sb_model = IPSBV2Model(
    path_unet_sb, ip_ckpt, aux_model, device=device, with_ip_mask_controller=True
)
print(f"Models ready in {time.time() - t0:.1f}s")

/Users/nguyenkz/Documents/code/CS2309.CH201/.venv/lib/python3.12/site-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/Users/nguyenkz/Documents/code/CS2309.CH201/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Using device: mps
Loading models (lần đầu có thể vài phút + tải HF)...


The config attributes {'sigma_max': None, 'sigma_min': None, 'timestep_type': 'discrete'} were passed to DDPMScheduler, but are not expected and will be ignored. Please verify your scheduler_config.json configuration file.
The config attributes {'decay': 0.9999, 'inv_gamma': 1.0, 'min_decay': 0.0, 'optimization_step': 110000, 'power': 0.6666666666666666, 'update_after_step': 0, 'use_ema_warmup': False} were passed to UNet2DConditionModel, but are not expected and will be ignored. Please verify your config.json configuration file.


Models ready in 30.2s


## Tham số test

Đổi `PRESET` hoặc gán trực tiếp `IMG_PATH`, `SRC_P`, `EDIT_P`.

In [4]:
PRESETS = {
    "woman": {
        "img": "assets/imgs_demo/woman_face.jpg",
        "src_p": "woman",
        "edit_p": "Taylor Swift",
    },
    "dog": {
        "img": "assets/imgs_demo/02.jpg",
        "src_p": "dog",
        "edit_p": "dog with mouth opened",
    },
}

PRESET = "dog"  # "woman" | "dog"

cfg = PRESETS[PRESET]
IMG_PATH = cfg["img"]
SRC_P = cfg["src_p"]
EDIT_P = cfg["edit_p"]

# Hyperparameter (ablation — giữ mặc định paper/repo)
SCALE_TA = 1.0
SCALE_EDIT = 0.2
SCALE_NON_EDIT = 1.0
CLAMP_RATE = 3.0
MASK_THRESHOLD = 0.5

print(f"Preset: {PRESET}")
print(f"  image: {IMG_PATH}")
print(f"  src:   {SRC_P!r}")
print(f"  edit:  {EDIT_P!r}")

Preset: dog
  image: assets/imgs_demo/02.jpg
  src:   'dog'
  edit:  'dog with mouth opened'


In [5]:
from PIL import Image
from torchvision.utils import save_image

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ModuleNotFoundError:
    HAS_MPL = False
    from IPython.display import display

assert Path(IMG_PATH).is_file(), f"Không thấy ảnh: {IMG_PATH}"

t0 = time.time()

# result = edit_image(
#     IMG_PATH,
#     SRC_P,
#     EDIT_P,
#     inverse_model,
#     aux_model,
#     ip_sb_model,
#     scale_ta=SCALE_TA,
#     scale_edit=SCALE_EDIT,
#     scale_non_edit=SCALE_NON_EDIT,
#     clamp_rate=CLAMP_RATE,
#     mask_threshold=MASK_THRESHOLD,
# )
# elapsed = time.time() - t0
# print(f"Edit {SRC_P!r} -> {EDIT_P!r} in {elapsed:.1f}s")

# out_name = f"nb_{PRESET}_{SRC_P}_to_{EDIT_P}.png".replace(" ", "_")
# out_path = OUTPUT_DIR / out_name
# save_image(result, out_path)
# print("Saved:", out_path)

# # Hiển thị input | output
# input_img = Image.open(IMG_PATH).convert("RGB").resize((512, 512))
# output_img = Image.open(out_path).convert("RGB")

# if HAS_MPL:
#     fig, axes = plt.subplots(1, 2, figsize=(10, 5))
#     axes[0].imshow(input_img)
#     axes[0].set_title(f"Input\nsrc: {SRC_P}")
#     axes[0].axis("off")
#     axes[1].imshow(output_img)
#     axes[1].set_title(f"Output ({elapsed:.1f}s)\nedit: {EDIT_P}")
#     axes[1].axis("off")
#     plt.tight_layout()
#     plt.show()
# else:
#     # Fallback nếu chưa cài matplotlib — chạy lại cell 2 (%pip install)
#     w, h = input_img.size
#     combined = Image.new("RGB", (w * 2, h))
#     combined.paste(input_img, (0, 0))
#     combined.paste(output_img, (w, 0))
#     display(combined)

### Upload ảnh riêng + chạy inference

1. Chạy các cell phía trên (đến **Load models**).
2. Chạy cell dưới: upload ảnh → nhập `src_p` / `edit_p` → **Áp dụng** → **Chạy SwiftEdit**.
3. Kết quả hiển thị trong ô output và lưu `results/notebook/`.

In [7]:
import sys

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
    import ipywidgets as widgets
    from IPython.display import display, clear_output

from PIL import Image
from torchvision.utils import save_image

try:
    import matplotlib.pyplot as plt

    HAS_MPL = True
except ModuleNotFoundError:
    HAS_MPL = False

upload_widget = widgets.FileUpload(
    accept=".jpg,.jpeg,.png",
    multiple=False,
    description="Ảnh",
)
src_widget = widgets.Text(
    value="dog",
    description="src_p",
    placeholder="mô tả ảnh gốc (có thể ngắn)",
    layout=widgets.Layout(width="420px"),
)
edit_widget = widgets.Text(
    value="dog with mouth opened",
    description="edit_p",
    placeholder="mô tả chỉnh sửa (bắt buộc)",
    layout=widgets.Layout(width="420px"),
)
btn_apply = widgets.Button(description="Áp dụng ảnh", button_style="info")
btn_run = widgets.Button(description="Chạy SwiftEdit", button_style="success")
status = widgets.HTML(value="<i>Chưa upload ảnh.</i>")
out = widgets.Output()


def _read_upload_bytes_and_name():
    """ipywidgets 8: value là tuple[dict]; ipywidgets 7: value là dict[name -> meta]."""
    val = upload_widget.value
    if not val:
        raise ValueError("Hãy chọn file ảnh (.jpg / .png) trước.")

    if isinstance(val, dict):
        name, meta = next(iter(val.items()))
        content = meta["content"]
        if hasattr(content, "tobytes"):
            content = content.tobytes()
        elif not isinstance(content, bytes):
            content = bytes(content)
        return name, content

    files = list(val)
    f0 = files[0]
    if isinstance(f0, dict):
        name = f0.get("name", "upload.jpg")
        content = f0["content"]
    else:
        name = getattr(f0, "name", "upload.jpg")
        content = f0.content
    if hasattr(content, "tobytes"):
        content = content.tobytes()
    elif not isinstance(content, bytes):
        content = bytes(content)
    return name, content


def _save_uploaded_image():
    name, data = _read_upload_bytes_and_name()
    ext = Path(name).suffix.lower()
    if ext not in {".jpg", ".jpeg", ".png"}:
        ext = ".jpg"
    path = OUTPUT_DIR / f"upload{ext}"
    path.write_bytes(data)
    return path


def on_apply(_):
    with out:
        clear_output(wait=True)
        try:
            path = _save_uploaded_image()
            global IMG_PATH, SRC_P, EDIT_P
            IMG_PATH = str(path)
            SRC_P = src_widget.value.strip()
            EDIT_P = edit_widget.value.strip()
            if not EDIT_P:
                raise ValueError("edit_p không được để trống.")
            preview = Image.open(path).convert("RGB")
            preview.thumbnail((320, 320))
            status.value = (
                f"<b>OK</b> — <code>{path.name}</code><br>"
                f"src: <code>{SRC_P}</code> → edit: <code>{EDIT_P}</code>"
            )
            print(f"IMG_PATH = {IMG_PATH}")
            print(f"SRC_P = {SRC_P!r}")
            print(f"EDIT_P = {EDIT_P!r}")
            display(preview)
        except Exception as e:
            status.value = f"<span style='color:red'>Lỗi: {e}</span>"


def on_run(_):
    with out:
        clear_output(wait=True)
        try:
            if "inverse_model" not in globals():
                raise RuntimeError("Chưa load model — chạy cell 'Load models' trước.")
            path = Path(IMG_PATH) if "IMG_PATH" in globals() and IMG_PATH else None
            if path is None or not path.is_file():
                path = _save_uploaded_image()
                globals()["IMG_PATH"] = str(path)
            SRC_P_local = src_widget.value.strip()
            EDIT_P_local = edit_widget.value.strip()
            if not EDIT_P_local:
                raise ValueError("edit_p không được để trống.")
            globals()["SRC_P"] = SRC_P_local
            globals()["EDIT_P"] = EDIT_P_local

            print(f"Đang edit {path.name}: {SRC_P_local!r} → {EDIT_P_local!r} ...")
            t0 = time.time()
            result = edit_image(
                str(path),
                SRC_P_local,
                EDIT_P_local,
                inverse_model,
                aux_model,
                ip_sb_model,
                scale_ta=SCALE_TA if "SCALE_TA" in globals() else 1.0,
                scale_edit=SCALE_EDIT if "SCALE_EDIT" in globals() else 0.2,
                scale_non_edit=SCALE_NON_EDIT if "SCALE_NON_EDIT" in globals() else 1.0,
                clamp_rate=CLAMP_RATE if "CLAMP_RATE" in globals() else 3.0,
                mask_threshold=MASK_THRESHOLD if "MASK_THRESHOLD" in globals() else 0.5,
            )
            elapsed = time.time() - t0
            safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in EDIT_P_local)[:40]
            out_path = OUTPUT_DIR / f"nb_upload_{safe}.png"
            save_image(result, out_path)
            print(f"Xong trong {elapsed:.1f}s — lưu: {out_path}")

            input_img = Image.open(path).convert("RGB").resize((512, 512))
            output_img = Image.open(out_path).convert("RGB")
            status.value = (
                f"<b>Hoàn tất</b> ({elapsed:.1f}s) — <code>{out_path.name}</code>"
            )

            if HAS_MPL:
                fig, axes = plt.subplots(1, 2, figsize=(10, 5))
                axes[0].imshow(input_img)
                axes[0].set_title(f"Input\n{SRC_P_local}")
                axes[0].axis("off")
                axes[1].imshow(output_img)
                axes[1].set_title(f"Output\n{EDIT_P_local}")
                axes[1].axis("off")
                plt.tight_layout()
                plt.show()
            else:
                w, h = input_img.size
                combined = Image.new("RGB", (w * 2, h))
                combined.paste(input_img, (0, 0))
                combined.paste(output_img, (w, 0))
                display(combined)
        except Exception as e:
            status.value = f"<span style='color:red'>Lỗi: {e}</span>"
            raise


btn_apply.on_click(on_apply)
btn_run.on_click(on_run)

display(
    widgets.VBox(
        [
            widgets.HTML("<b>Upload + prompt</b>"),
            upload_widget,
            src_widget,
            edit_widget,
            widgets.HBox([btn_apply, btn_run]),
            status,
            out,
        ]
    )
)